# Data Engineering Pipeline for Supplement Experiments

1001-Experiments makes personalized supplements tailored to individual health needs.

1001-Experiments aims to enhance personal health by using data from wearable devices and health apps.

This data, combined with user feedback and habits, is used to analyze and refine the effectiveness of the supplements provided to the user through multiple small experiments.

The data engineering team at 1001-Experiments plays a crucial role in ensuring the collected health and activity data from thousands of users is accurately organized and integrated with the data from supplement usage. 

This integration helps 1001-Experiments provide more targeted health and wellness recommendations and improve supplement formulations.


## Task

1001-Experiments currently has the following four datasets with four months of data:
 - "user_health_data.csv" which logs daily health metrics, habits and data from wearable devices,
 - "supplement_usage.csv" which records details on supplement intake per user,
 - "experiments.csv" which contains metadata on experiments, and
 - "user_profiles.csv" which contains demographic and contact information of the users.

Each dataset contains unique identifiers for users and/or their supplement regimen.

The developers and data scientsits currently manage code that cross-references all of these data sources separately, which is cumbersome and error-prone.

Your manager has asked you to write a Python function that cleans and merges these datasets into a single dataset.

The final dataset should provide a comprehensive view of each user's health metrics, supplement usage, and demographic information.

- To test your code, your manager will run only the code `merge_all_data('user_health_data.csv', 'supplement_usage.csv', 'experiments.csv', 'user_profiles.csv')`
- Your `merge_all_data` function must return a DataFrame, with columns as described below.
- All columns must accurately match the descriptions provided below, including names.


## Data

The provided data is structured as follows:

![database schema](schema.png)

The function you write should return data as described below.

There should be a unique row for each daily entry combining health metrics and supplement usage.

Where missing values are permitted, they should be in the default Python format unless stated otherwise.

| Column Name        | Description |
|--------------------|-------------|
| user_id            | Unique identifier for each user. </br>There should not be any missing values. |
| date               | The date the health data was recorded or the supplement was taken, in date format. </br>There should not be any missing values. |
| email              | Contact email of the user. </br>There should not be any missing values. |
| user_age_group  | The age group of the user, one of: 'Under 18', '18-25', '26-35', '36-45', '46-55', '56-65', 'Over 65' or 'Unknown' where the age is missing.|
| experiment_name    | Name of the experiment associated with the supplement usage. </br>Missing values for users that have user health data only is permitted. |
| supplement_name    | The name of the supplement taken on that day. Multiple entries are permitted. </br>Days without supplement intake should be encoded as 'No intake'. |
| dosage_grams       | The dosage of the supplement taken in grams. Where the dosage is recorded in mg it should be converted by division by 1000.</br>Missing values for days without supplement intake are permitted. |
| is_placebo         | Indicator if the supplement was a placebo (true/false). </br>Missing values for days without supplement intake are permitted. |
| average_heart_rate | Average heart rate as recorded by the wearable device. </br>Missing values are permitted. |
| average_glucose    | Average glucose levels as recorded on the wearable device. </br>Missing values are permitted. |
| sleep_hours        | Total sleep in hours for the night preceding the current day’s log. </br>Missing values are permitted. |
| activity_level     | Activity level score between 0-100. </br>Missing values are permitted. |

In [1]:
#Load datasets
import pandas as pd
import numpy as np

def extract(user_health_file, supplement_usage_file, experiments_file, user_profiles_file):
    user_health_data = pd.read_csv(user_health_file)
    supplement_usage = pd.read_csv(supplement_usage_file)
    experiments = pd.read_csv(experiments_file)
    user_profiles = pd.read_csv(user_profiles_file)
    return user_health_data, supplement_usage, experiments, user_profiles

In [2]:
# Transform datasets
def merging(user_health_data, supplement_usage, experiments, user_profiles):

    # Convert the date columns to datetime format
    user_health_data['date'] = pd.to_datetime(user_health_data['date'])
    # Extract numeric values from sleep_hours column (remove 'h' and 'H')
    user_health_data ['sleep_hours'] = user_health_data ['sleep_hours'].str.replace('h', '', case=False).astype(float)

    # Clean and prepare supplement data
    supplement_usage['date'] = pd.to_datetime(supplement_usage['date'])

    # Convert dosage from mg to grams
    supplement_usage['dosage_grams'] = supplement_usage['dosage'] / 1000

    # Merge supplement_usage with experiments
    supplement_with_exp = supplement_usage.merge(experiments[['experiment_id', 'name']], how='left', on='experiment_id')
    supplement_with_exp = supplement_with_exp.rename(columns={'name': 'experiment_name'})

      
    # Create age groups function
    def categorize_age(age):
        if pd.isna(age):
            return 'Unknown'
        elif age < 18:
            return 'Under 18'
        elif age <= 25:
            return '18-25'
        elif age <= 35:
            return '26-35'
        elif age <= 45:
            return '36-45'
        elif age <= 55:
            return '46-55'
        elif age <= 65:
            return '56-65'
        else:
            return 'Over 65'

    # Apply age categorization
    user_profiles['user_age_group'] = user_profiles['age'].apply(categorize_age)
    
    user_profiles['user_age_group'].fillna('Unknown', inplace=True)

    #Merge health data with user profile
    health_profile = user_health_data.merge(
        user_profiles[['user_id', 'email', 'user_age_group']], 
        on='user_id', 
        how='left'
    )
     # Merge with supplement data (left join to keep all health records)
    merged_data = health_profile.merge(
        supplement_with_exp[['user_id', 'date', 'experiment_name', 'supplement_name', 
                           'dosage_grams', 'is_placebo']], 
        on=['user_id', 'date'], 
        how='left'
    )

    # Handle days without supplement intake
    merged_data['supplement_name'] = merged_data['supplement_name'].fillna('No intake')

    # Ensure all required columns are present and in correct order
    final_columns = [
        'user_id', 'date', 'email', 'user_age_group', 'experiment_name',
        'supplement_name', 'dosage_grams', 'is_placebo', 'average_heart_rate',
        'average_glucose', 'sleep_hours', 'activity_level'
    ]
    
    # Reorder columns and return
    merged_data = merged_data[final_columns]
    
    return merged_data

In [3]:
def transform(merged_data):
    # Transform columns
    
    
# Strip whitespace from all string columns
    for col in merged_data.select_dtypes(include='string').columns:
        merged_data[col] = merged_data[col].str.strip()



    # Set is_placebo based on supplement_name
    merged_data['is_placebo'] = merged_data['supplement_name'].apply(
        lambda x: True if x == 'Placebo' else (np.nan if x == 'No intake' else False)
    )

  
    #Transforming
    merged_data['is_placebo'] = merged_data['is_placebo'].astype('string')
    merged_data['experiment_name'] = merged_data['experiment_name'].astype('string')
    merged_data['user_age_group'] = merged_data['user_age_group'].astype('string')
    merged_data['activity_level'] = merged_data['activity_level'].astype('int')
    merged_data['email'] = merged_data['email'].astype('string')
    merged_data['user_id'] = merged_data['user_id'].astype('string')
    merged_data['supplement_name'] = merged_data['supplement_name'].astype('string')
        
    transformed_data = merged_data
    return transformed_data

In [4]:
# Main function
def merge_all_data(user_health_file, supplement_usage_file, experiments_file, user_profiles_file):
    user_health_data, supplement_usage, experiments, user_profiles = extract(
        user_health_file, supplement_usage_file, experiments_file, user_profiles_file
    )
    merged_data = merging(user_health_data, supplement_usage, experiments, user_profiles)
    transformed_data = transform(merged_data)
    return transformed_data


In [5]:
# Result
merge_all_data(
    'user_health_data.csv',
    'supplement_usage.csv',
    'experiments.csv',
    'user_profiles.csv'
)

,user_id,date,email,user_age_group,experiment_name,supplement_name,dosage_grams,is_placebo,average_heart_rate,average_glucose,sleep_hours,activity_level
0,c6ae338a-9f95-481c-a88d-24a58bc8fc71,2018-01-31,hi_1@example.com,Over 65,Sleep Quality,Zinc,0.149315,False,93.055612,70.089910,8.8,1
1,c6ae338a-9f95-481c-a88d-24a58bc8fc71,2018-02-28,hi_1@example.com,Over 65,<NA>,No intake,NaN,<NA>,88.059964,78.411148,8.0,3
2,c6ae338a-9f95-481c-a88d-24a58bc8fc71,2018-03-31,hi_1@example.com,Over 65,Endurance,Magnesium,0.239949,False,78.373746,107.418818,11.9,1
3,c6ae338a-9f95-481c-a88d-24a58bc8fc71,2018-03-31,hi_1@example.com,Over 65,Focus,Placebo,0.416022,True,78.373746,107.418818,11.9,1
4,c6ae338a-9f95-481c-a88d-24a58bc8fc71,2018-04-30,hi_1@example.com,Over 65,Recovery,Placebo,0.324913,True,62.204061,117.259092,5.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2716,9c083dc3-b789-4cee-87fb-9887ff4b4e29,2018-04-30,contact499@myemail.com,46-55,<NA>,No intake,NaN,<NA>,88.575502,121.997505,7.9,3
2717,2f2aa322-d2b2-4c88-aeb6-e49e768521f3,2018-01-31,hello_500@myemail.com,46-55,<NA>,No intake,NaN,<NA>,96.665724,73.559778,9.9,1
2718,2f2aa322-d2b2-4c88-aeb6-e49e768521f3,2018-02-28,hello_500@myemail.com,46-55,Strength,Vitamin C,0.280467,False,88.219866,85.478628,7.9,4
2719,2f2aa322-d2b2-4c88-aeb6-e49e768521f3,2018-03-31,hello_500@myemail.com,46-55,<NA>,No intake,NaN,<NA>,86.871171,89.198330,7.9,3
